## Embeddings (dense ViT → patch + cls token parquet)

Stream chips with **per-chip view batching**, write `ssl4eo_cls-patch.parquet`.

Core logic: **`gee/embed_cls_patch.py`** (`DenseEmbedConfig`, `collect_cls_patch_embedding_table`, …).

**Train a probe** in `train_probe.ipynb` (`EMBEDDING_STRATEGY="cls_patch"`).

**Feature column order (ViT-S/16 → 384-d CLS + 384-d patch):** each row is class token **then** one jitter-selected patch token: `cls0`…`cls383`, then `spatial0`…`spatial383`. This matches `np.concatenate([cls, patch], axis=1)` in `embed_cls_patch.cls_patch_feature_batch` and the first/second half split in `model_library.MLP_with_targeted_dropout`.


In [ ]:
from pathlib import Path
import sys

import geopandas as gpd
import pandas as pd

parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

from embed_cls_patch import DenseEmbedConfig

%load_ext autoreload
%autoreload 2

data_parent = Path("../data")
data_timestamp = "2026-03-24T23:15"

cfg = DenseEmbedConfig(
    train_n_views = 8,
    eval_n_views = 8
)

data_dir = data_parent / f"training_patches{data_timestamp}"
collected_locations_path = (
    data_parent / "sampling_locations" / f"collected_locations{data_timestamp}.geojson"
)


In [ ]:
collected_locations = gpd.read_file(collected_locations_path)
print("Input labeled datasets:")
set(collected_locations.source_file)


### Embedding inference

Uses ``InferenceEngine.embed_dense`` (via ``gee/embed_cls_patch.py``) with **batched viewports per super-chip**. Keep ``cfg.embedding_batch_size`` ≥ ``cfg.train_n_views``.

``InferenceEngine`` still carries GEE + Keras classifier; this notebook only uses the ViT path.


In [ ]:
from embed_cls_patch import (
    collect_cls_patch_embedding_table,
    feature_column_names_cls_patch,
    make_embedding_engine,
)


In [ ]:
embedding_engine = make_embedding_engine(cfg)
print(f"Device: {embedding_engine.device}")

output_dim = embedding_engine.embed_model.norm.normalized_shape[0]
feature_columns = feature_column_names_cls_patch(output_dim)

In [ ]:
# Optional: restrict to some GeoJSON sources (stems must match folders under data_dir).
# example:
# stems = {Path(p).stem for p in collected_locations["source_file"] if "ACA" in str(p)}
# source_files = [data_parent / "sampling_locations" / f"{s}.geojson" for s in stems]
source_files = None  # all sources under data_dir; or set list of paths / stems above

splits_to_embed = ["val", "test1", "test2", "test3", "train"]

df = collect_cls_patch_embedding_table(
    cfg,
    data_dir,
    embedding_engine,
    splits=splits_to_embed,
    source_files=source_files,
    feature_col_names=feature_columns,
)

df[["label", "split"]].value_counts()

In [ ]:
parquet_path = data_dir / "ssl4eo_cls-patch.parquet"
df.to_parquet(parquet_path, index=False)
print(f"Wrote {len(df)} rows to {parquet_path}")
